In [2]:
import sys
sys.path.append("../../")
import json
import pandas as pd
from pathlib import Path
import time

from prompts.metadata_extract_prompt import metadata_extract_prompt
from evaluation.eval import product_load_tests
from models.generation import openai_generate, ollama_generate


In [9]:
tests = product_load_tests()

In [11]:
METADATA_FIELDS = [
      "brand",
      "color",
      "category",
      "top_type",
      "bottom_type",
      "dupatta",
      "pattern",
      "shape",
      "neck",
      "sleeves",
      "length",
      "hemline",
      "top_fabric",
      "bottom_fabric",
      "dupatta_fabric",
      "occasion",
      "wash_care",
      "price",
  ]

In [12]:
def normalize_metadata(metadata: dict) -> dict:
    normalized = {}

    for field in METADATA_FIELDS:
        value = metadata.get(field)

        if field == "price":
            price = value or {"min": 0, "max": "inf"}
            normalized[field] = {
                "min": price.get("min", 0),
                "max": price.get("max", "inf"),
            }
        else:
            normalized[field] = sorted(set(value or []))

    return normalized

In [13]:
def compare_metadata(expected: dict, predicted: dict) -> dict:
    expected = normalize_metadata(expected)
    predicted = normalize_metadata(predicted)

    field_scores = {}
    for field in METADATA_FIELDS:
        field_scores[field] = 1 if expected[field] == predicted[field] else 0

    exact_match = 1 if expected == predicted else 0
    field_accuracy = sum(field_scores.values()) / len(METADATA_FIELDS)

    return {
        "exact_match": exact_match,
        "field_accuracy": field_accuracy,
        **field_scores,
    }

In [ ]:
rows = []

for test in tests[:10]:
    prompt = metadata_extract_prompt.render(query=test.question)

    last_error = None
    response = None

    for attempt in range(3):
        try:
            response = openai_generate(prompt)
            break
        except Exception as e:
            last_error = e
            time.sleep(2)

    if response is None:
        rows.append({
            "question": test.question,
            "expected_metadata": test.metadata,
            "predicted_metadata": None,
            "exact_match": 0,
            "field_accuracy": 0,
            "error": str(last_error),
        })
        continue

    predicted_metadata = json.loads(response)
    scores = compare_metadata(test.metadata, predicted_metadata)

    rows.append({
        "question": test.question,
        "expected_metadata": test.metadata,
        "predicted_metadata": predicted_metadata,
        "error": None,
        **scores,
    })

In [15]:
report_df = pd.DataFrame(rows)
report_df[["question", "exact_match", "field_accuracy"]]

,question,exact_match,field_accuracy
0,Show me Regular trousers within a budget of 2000.,0,0.888889
1,"Can you find something similar to ""Athena Wome...",0,0.666667
2,"Show me products like ""InWeave Women Red Print...",0,0.666667
3,I need product with a Woven design pattern in ...,0,0.888889
4,I'm looking for products from MANGO for everyd...,0,0.833333
5,What do you have from Sweet Dreams in Playsuit...,0,0.722222
6,I'm looking for Kurta set by Vishudh.,0,0.833333
7,I'm looking for A-line by Ancestry.,0,0.833333
8,"Can you find something similar to ""FASHOR Wome...",0,0.444444
9,Show me Blue Tailored jacket from Darzi.,0,0.555556


In [16]:
summary_df = pd.DataFrame([
    {
        "samples": len(report_df),
        "exact_match": report_df["exact_match"].mean(),
        "field_accuracy": report_df["field_accuracy"].mean(),
        **{field: report_df[field].mean() for field in METADATA_FIELDS},
    }
]).round(3)

summary_df

,samples,exact_match,field_accuracy,brand,color,category,top_type,bottom_type,dupatta,pattern,...,neck,sleeves,length,hemline,top_fabric,bottom_fabric,dupatta_fabric,occasion,wash_care,price
0,10,0.0,0.733,1.0,0.9,0.3,1.0,1.0,0.9,0.7,...,0.8,0.6,0.6,0.4,0.5,1.0,1.0,0.3,0.3,1.0


In [17]:
report_path = Path("../../reports/product_metadata_extraction_evaluation.md")
report_path.parent.mkdir(parents=True, exist_ok=True)
summary_row = summary_df.iloc[0]
report_text = [
    "# Product Metadata Extraction Evaluation",
    "",
    "## Overview",
    "",
    f"- Samples: {int(summary_row['samples'])}",
    f"- Exact match: {summary_row['exact_match']:.3f}",
    f"- Field accuracy: {summary_row['field_accuracy']:.3f}",
    "",
    "## Per-field Accuracy",
    "",
]

for field in METADATA_FIELDS:
    report_text.append(f"- {field}: {summary_row[field]:.3f}")

report_path.write_text("\n".join(report_text), encoding="utf-8")
report_path

PosixPath('../../reports/product_metadata_extraction_evaluation.md')